No se importa de inference.funcs dado que el notebook almacena una cache del archivo y si se realizan modificaciones sobre inference.funcs estas no se ven impactadas al volver a hacer el import.

In [ ]:
from pathlib import Path
from llama_stack_client.lib.agents.agent import Agent
from llama_stack_client.types.agents.turn import Turn
from llama_stack_client.types import Document, UserMessage, QueryChunksResponse
from llama_stack_client.types.agent_create_params import AgentConfig
#from client import client, MODEL, VECTOR_DB_ID, EMBEDDING_MODEL, SESSION_NAME
from os import listdir
from os.path import isfile, join
from pprint import pprint

from llama_stack_client import LlamaStackClient

HOST = "20.72.80.241"
PORT = 5001
MODEL = "meta-llama/Llama-3.2-3B-Instruct"
VECTOR_DB_ID = 'accounts_no_quotes'
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
SESSION_NAME = 'test-rag-accounts'

client = LlamaStackClient(
    base_url=f"http://{HOST}:{PORT}",
)

def create_rag():
    vector_providers = [
        provider for provider in client.providers.list() if provider.api == "vector_io"
    ]
    if not vector_providers:
        raise Exception('No available vector_io providers.')

    selected_vector_provider = vector_providers[0]

    # Create new rag
    client.vector_dbs.register(
        vector_db_id=VECTOR_DB_ID,
        embedding_model=EMBEDDING_MODEL,
        embedding_dimension=384,
        provider_id=selected_vector_provider.provider_id,
    )

def clear_rag():
    # Delete existing rag
    try:
        client.vector_dbs.unregister(VECTOR_DB_ID)
        global fileid
        fileid = 1
    except Exception:
        pass

    create_rag()
    return

def get_mime_type(file_extension: str):
    mime_type = 'text/plain'

    if file_extension == '.csv':
        mime_type = 'text/csv'
    elif file_extension == '.json':
        mime_type = 'application/json'

    return mime_type


def load_rag(documents: list[dict], truncate_rag: bool = False, chunk_size_in_tokens: int = 512) -> bool:

    if 'fileid' not in globals():
        global fileid
        fileid = 1

    parsed_documents = []

    if truncate_rag:
        clear_rag()

    # Chunk size
    n = 100
    batchs = [documents[i:i + n] for i in range(0, len(documents), n)]

    try:
        for batch in batchs:
            for document in batch:
                parsed_doc = Document(
                    document_id=f'{document["file"][:40]}{document["file_extension"]}-{fileid}',
                    content=document["content"],
                    mime_type=get_mime_type(document["file_extension"]),
                    metadata={},
                )
                parsed_documents.append(parsed_doc)
                fileid += 1
                
            # Insert documents using the RAG tool
            client.tool_runtime.rag_tool.insert(
                documents=parsed_documents,
                vector_db_id=VECTOR_DB_ID,
                chunk_size_in_tokens=chunk_size_in_tokens,
                timeout=1200
            )

    except Exception as e:
        print(e)
        return False
    return True

def load_files(folder: str, chunk_size_in_tokens: int = 512):

    documents = []
    errors = []

    all_file_paths = [join(folder, f) for f in listdir(folder) if isfile(join(folder, f))]

    for file_path in all_file_paths:
        try:
            file_name = Path(file_path).stem
            file_extension = Path(file_path).suffix
            file = open(file_path, "r", encoding='utf-8')
            content = file.read()
            file.close()
            document = {'file': file_name, 'content': content, 'file_extension': file_extension}
            documents.append(document)
        except Exception:
            errors.append(f'Error loading file {file_path}. Skipping')
            continue

    loaded_rag = load_rag(documents=documents, truncate_rag=False, chunk_size_in_tokens=chunk_size_in_tokens)
    if loaded_rag:
        if len(errors) > 0:
            print("\n".join(errors))
    return

def inference_with_rag(prompt: str, root_prompt: str = None, enable_rag: bool = True) -> Turn:
    available_shields = [shield.identifier for shield in client.shields.list()]

    toolgroups = []

    if enable_rag:
        rag_tool = {
                "name": "builtin::rag",
                "args": {"vector_db_ids": [VECTOR_DB_ID]},
                }
        toolgroups.append(rag_tool)

    root_prompt = "" if root_prompt == None else root_prompt
    agent_config = AgentConfig(
        model=MODEL,
        instructions=root_prompt,
        sampling_params={
            "strategy": {"type": "greedy"},
        },
        toolgroups=toolgroups,
        tool_choice="auto",
        tool_prompt_format="python_list",
        input_shields=available_shields if available_shields else [],
        output_shields=available_shields if available_shields else [],
        enable_session_persistence=False,
    )

    agent = Agent(client, agent_config)
    session_id = agent.create_session(SESSION_NAME)

    response = agent.create_turn(
            messages=[
                UserMessage(role='user', content=prompt),
            ],
            session_id=session_id,
            stream=False
    )

    return response

def get_context(response: Turn) -> str:
    try:
        context_map = map(lambda x: x.text, response.input_messages[0].context)
        context = ''.join(list(context_map))
    except Exception:
        #print('Error al obtener el contexto. Revisar si hay algo cargado en el RAG.')
        return
    return context

def get_output(response: Turn) -> str:
    return response.output_message.content

def inference_chat(prompt: str, root_prompt: str) -> list[str]:
    response = inference_with_rag(prompt, root_prompt)
    return [get_output(response), get_context(response)]

def query_rag(query: str) -> QueryChunksResponse:
    response = client.vector_io.query(vector_db_id=VECTOR_DB_ID,
                       query=query)
    
    return response

def prompt_generator(account: str) -> str:
    rag_response = query_rag(account)
    rag_content = rag_response.chunks[0].content

    prompt = f"""Dispones del siguiente contexto ```{rag_content}```
    
    Tu tarea es responder a que cuenta corresponde la siguiente: ```{account}```
    """
    return prompt

In [14]:
client.models.list()

APIConnectionError: Connection error.

In [ ]:
client.

False

In [ ]:
# Simple inference
#clear_rag()
response, context = inference_chat(
                    prompt="Hola, como estas?",
                    root_prompt=''
)

print(f'Response: {response}\n\nContext: {context}')

Response: Hola! Estoy bien, gracias. ¿Y tú? ¿En qué puedo ayudarte hoy?

Context: None


In [ ]:
# Inference using root prompt
ROOT_PROMPT = '''Tu tarea es indicar a que cuenta corresponde el prompt que introdujo el usuario ayudandote con tu contexto.'''

response, context = inference_chat(
                        prompt='"Propiedades y posesiones"',
                        root_prompt= ROOT_PROMPT
                    )

print(f'Response: {response}\n\nContext: {context}')

APITimeoutError: Request timed out.

In [2]:
ROOT_FOLDER = './rag_files'

### Load one file per account ###

In [25]:
FOLDER = f'{ROOT_FOLDER}/3_one_file_per_account'
clear_rag()
load_files(FOLDER)

Request timed out.


In [26]:
response, context = inference_chat(
                        prompt='Propiedades y posesiones',
                        root_prompt= 'Indicar a que cuenta corresponde el prompt del usuario.'
                    )

print(f'Response: {response}\n\nContext: {context}')

Response: El prompt "Propiedades y posesiones" puede corresponder a varios temas relacionados con la propiedad intelectual, material o inmaterial de una persona o entidad. A continuación, te presento algunas posibles interpretaciones:

1. **Propiedades inmobiliarias**: se refiere a bienes raíces, edificios, terrenos, vehículos, etc.
2. **Propiedad intelectual**: se trata de derechos legales que otorgan a una persona o entidad el control sobre creaciones intelectuales como patentes, marcas registradas, derechos de autor, diseños industriales, etc.
3. **Posesiones personales**: se refiere a objetos personales que pertenecen a una persona, como joyas, ropa, muebles, arte, colecciones, etc.
4. **Propiedades financieras**: se trata de activos y pasivos que una persona o entidad tiene en su cuenta bancaria, inversiones, bienes raíces, etc.
5. **Propiedad digital**: se refiere a derechos digitales sobre contenido, como software, aplicaciones, bases de datos, dominios, etc.

Estas son solo alg

### Reduced one file per account ###

In [ ]:
FOLDER = f'{ROOT_FOLDER}/3_one_file_per_account_redux'
clear_rag()
load_files(folder=FOLDER, chunk_size_in_tokens=64)

In [ ]:
response, context = inference_chat(
                        prompt='Propiedades y posesiones',
                        root_prompt= 'Indicar a que cuenta corresponde el prompt del usuario.'
                    )

print(f'Response: {response}\n\nContext: {context}')

### Load one file with all accounts ###

In [ ]:
FOLDER = f'{ROOT_FOLDER}/4_one_large_file'
clear_rag()
load_files(FOLDER, chunk_size_in_tokens=64)

### Load one file with all accounts - no quotes ###

In [ ]:
FOLDER = f'{ROOT_FOLDER}/5_one_large_file_no_quotes'
clear_rag()
load_files(FOLDER, chunk_size_in_tokens=64)

### Load one file with all accounts - no quotes - multi ###

In [17]:
FOLDER = f'{ROOT_FOLDER}/6_one_large_file_no_quotes_multi'
clear_rag()
load_files(FOLDER, chunk_size_in_tokens=128)

### Testing ###

In [22]:
response, context = inference_chat(
                        prompt='Propiedades y posesiones',
                        root_prompt= 'Indicar a que cuenta corresponde el prompt del usuario.'
                    )

print(f'Response: {response}\n\nContext: {context}')

Response: El prompt del usuario se refiere a la cuenta contable de "Propiedades, mobiliario y equipo".

Context: Here are the retrieved documents for relevant context:
=== START-RETRIEVED-CONTEXT ===
id:accounts.txt-1; content:ibles corresponde a la cuenta Activos por derechos de uso de propiedades, mobiliario y equipo
Activos por derechos de uso de activos tangibles corresponde a la cuenta Activos por derechos de uso de propiedades, mobiliario y equipo

Propiedades sin edificaciones corresponde a la cuenta Terrenos
Propiedades sin edificaciones corresponde a la cuenta Terrenos
Bienes inmuebles corresponde a la cuenta Terrenos
Bienes inmuebles corresponde a la cuenta Terrenos
Parcelas corresponde a la cuenta Terrenos
Parcelas corresponde a la cuentaid:accounts.txt-1; content: equipamiento corresponde a la cuenta Propiedades, mobiliario y equipo
Activo tangible corresponde a la cuenta Propiedades, mobiliario y equipo
Activo tangible corresponde a la cuenta Propiedades, mobiliario y equi

In [58]:
#"Compromisos vencidos" -> "Adeudos vencidos"
# "path": "ACTIVO > Otras cuentas por cobrar > Deudores diversos > Adeudos vencidos"

account = '¿Se encuentra en el contexto la cuenta "Compromisos vencidos"?'
account = '¿Se encuentra en el contexto la cuenta "Estado de intere publico"?'
account = '¿Se encuentra en el contexto la cuenta "Préstamos otorgados por el INFONAVIT"?'

account = '¿Se encuentra en el contexto la cuenta "Compromisos vencidos"?'
account = 'Cuentas por cobrar representan derechos de cobro de la empresa, es decir dinero que terceros le deben a la empresa. ¿La cuenta de "Otras cuentas por cobrar, Deudores varios, Compromisos vencidos" corresponde a un Pasivo o Activo?'

account = 'Las siguientes son cuentas de Activos: Efectivo y equivalentes de efectivo; Cuentas de margen (instrumentos financieros derivados); Inversiones en instrumentos financieros; Deudores por reporto; Préstamo de valores; Instrumentos financieros derivados; Ajustes de valuación por cobertura de activos financieros; Cartera de crédito con riesgo de crédito etapa 1; Cartera de crédito con riesgo de crédito etapa 2; Cartera de crédito con riesgo de crédito etapa 3; Cartera de crédito valuada a valor razonable; Estimación preventiva para riesgos crediticios; Derechos de cobro adquiridos (créditos deteriorados); Estimación preventiva para riesgos crediticios derivada de derechos de cobro adquiridos (créditos deteriorados); Activos virtuales; Beneficios por recibir en operaciones de bursatilización; Otras cuentas por cobrar; Estimación de pérdidas crediticias esperadas; Bienes adjudicados; Estimación de bienes adjudicados; Activos de larga duración mantenidos para la venta o para distribuir a los propietarios; Activos relacionados con operaciones discontinuadas; Pagos anticipados y otros activos; Propiedades, mobiliario y equipo; Depreciación acumulada de propiedades, mobiliario y equipo; Activos por derechos de uso de propiedades, mobiliario y equipo; Depreciación de activos por derechos de uso de propiedades, mobiliario y equipo; Inversiones permanentes; Activo por impuestos a la utilidad diferidos; Activos intangibles; Amortización acumulada de activos intangibles; Activos por derechos de uso de activos intangibles; Amortización de activos por derechos de uso de activos intangibles; Crédito mercantil; ¿La cuenta de **Deudas por cobrar de otro tipo, Deudores varios, Adeudos Vencidos** a que cuenta de Activos corresponderia?'

ROOT_PROMPT = 'Tu eres un asesor de contabilidad y finanzas. Te especializas en Libros contables de entidades financieras en Latinoamérica'

#prompt = prompt_generator(account)
response = inference_with_rag(account, root_prompt=ROOT_PROMPT, enable_rag=False)
print(get_output(response), get_context(response))

La cuenta de "Deudas por cobrar de otro tipo, Deudores varios, Adeudos Vencidos" corresponde a la cuenta de Activos "Otras cuentas por cobrar".

En general, las cuentas por cobrar se clasifican en dos categorías principales:

1. Cuentas por cobrar de otro tipo (o deudores varios): se refiere a deudas que no se han cobrado y que corresponden a personas o entidades diferentes a la empresa.
2. Cuentas por cobrar de la empresa (o adeudos vencidos): se refiere a deudas que no se han cobrado y que corresponden a la propia empresa.

En el caso de la cuenta "Deudas por cobrar de otro tipo, Deudores varios, Adeudos Vencidos", se refiere a deudas que no se han cobrado y que corresponden a personas o entidades diferentes a la empresa, por lo que se clasifica como una cuenta por cobrar de otro tipo.

Por lo tanto, la cuenta de "Deudas por cobrar de otro tipo, Deudores varios, Adeudos Vencidos" corresponde a la cuenta de Activos "Otras cuentas por cobrar". None


In [42]:
account = 'Se encuentra en el contexto la cuenta Compromisos vencidos'
#
#"input": "Compromisos vencidos",
#"expected_answer": " Adeudos vencidos"
response = response = client.vector_io.query(vector_db_id=VECTOR_DB_ID,
                       query=account)
pprint(response.to_dict())

{'chunks': [{'content': 'uenta de compromiso comunitario corresponde a la '
                        'cuenta De interés social\n'
                        'Cuenta de compromiso comunitario corresponde a la '
                        'cuenta De interés social\n'
                        'Cuenta de contribución social corresponde a la cuenta '
                        'De interés social\n'
                        'Cuenta de contribución social corresponde a la cuenta '
                        'De interés social\n'
                        'Cuenta de apoyo a la comunidad corresponde a la '
                        'cuenta De interés social\n'
                        'Cuenta de apoyo a la comunidad corresponde a la '
                        'cuenta De interés social\n'
                        'Cuenta de impacto social corresponde a la cuenta De '
                        'interés social\n'
                        'Cuenta de impacto social corresponde a',
             'metadata': {'document_id': 'a

### Unit Tests ###

In [57]:
ROOT_PROMPT = '''Tu tarea es indicar a que cuenta corresponde el prompt que introdujo el usuario usando la informacion en tu contexto.
Para responder utiliza el siguiente formato:
```
La cuenta corresponde a: "{nombre de cuenta}"
```'''

In [68]:
account = 'Cuentas por cobrar de terceros'
prompt = f'A que cuenta corresponde "{account}"'


account = 'Adeudos de terceros'
prompt = f'A que cuenta corresponde "{account}"'

account = 'Obligaciones por cuenta de terceros'
prompt = f'A que cuenta corresponde "{account}"'


response, context = inference_chat(
                        prompt=prompt,
                        root_prompt=''
)

print(f'Response: {response}\n\nContext: {context}')

Response: La cuenta "Obligaciones por cuenta de terceros" corresponde a la cuenta "Aceptaciones por cuenta de terceros".

Context: Here are the retrieved documents for relevant context:
=== START-RETRIEVED-CONTEXT ===
id:accounts.txt-1; content: cuenta de terceros"
"Deudas a favor de terceros" corresponde a la cuenta "Aceptaciones por cuenta de terceros"
"Obligaciones por cuenta de terceros" corresponde a la cuenta "Aceptaciones por cuenta de terceros"
"Pasivos de terceros" correspondeid:accounts.txt-1; content: sin refinanciamiento"

"Cuentas por cobrar de terceros" corresponde a la cuenta "Aceptaciones por cuenta de terceros"
"Adeudos de terceros" corresponde a la cuenta "Aceptaciones por cuenta de terceros"
"Deudas a favor de terceros" correspondeid:accounts.txt-1; content: por cuenta de terceros"
"Aceptaciones de terceros" corresponde a la cuenta "Aceptaciones por cuenta de terceros"
"Aceptaciones por cuenta ajena" corresponde a la cuenta "Aceptaciones por cuenta de terceros"
"Acep

In [69]:
prompt = 'Cuentas por cobrar de terceros'

response, context = inference_chat(
                        prompt=prompt,
                        root_prompt='Busca taxativamente en el RAG el prompt del usuario. Informa en funcion del resultado del contexto a que cuenta corresponde el prompt del usuario.'
)

print(f'Response: {response}\n\nContext: {context}')

Response: Después de analizar el contexto proporcionado, puedo identificar que el prompt del usuario es:

"Cuentas por cobrar de terceros"

Este prompt se relaciona con la cuenta "Instrumentos financieros para cobrar principal e interés restringidos o dados en garantía en operaciones de reporto", ya que se menciona explícitamente en el contexto proporcionado.

Context: Here are the retrieved documents for relevant context:
=== START-RETRIEVED-CONTEXT ===
id:accounts.txt-1; content:"Cambio por revalorización de los beneficios definidos a los trabajadores" corresponde a la cuenta "Incremento por actualización de la remedición de beneficios definidos a los empleados (1)"
"Incremento por ajuste de la valuación de beneficios definidos a empleados" corresid:accounts.txt-1; content: a la cuenta "Beneficios por terminación"
"Utilidades por cese" corresponde a la cuenta "Beneficios por terminación"
"Resultados por finalización" corresponde a la cuenta "Beneficios por terminación"
"Ingresos por 

In [70]:
response, context = inference_chat(
                        prompt='''Tu tarea es indicar a que cuenta corresponde el prompt que introdujo el usuario usando la informacion en tu contexto.
Para responder utiliza el siguiente formato:
```
La cuenta corresponde a: "{nombre de cuenta}"
```''',
                        root_prompt='Cuentas por cobrar de terceros'
)

print(f'Response: {response}\n\nContext: {context}')

Response: La cuenta corresponde a: Cuentas por cobrar condicionadas

Context: Here are the retrieved documents for relevant context:
=== START-RETRIEVED-CONTEXT ===
id:accounts.txt-1; content:" corresponde a la cuenta "Fideicomisos públicos de contratación"

"Cuentas Varias" corresponde a la cuenta "Otros"
"Cuentas Diversas" corresponde a la cuenta "Otros"
"Otras Cuentas" corresponde a la cuenta "Otrosid:accounts.txt-1; content: en cuentas por pagar" corresponde a la cuenta "Créditos en cuenta corriente"
"Créditos en cuentas corrientes" corresponde a la cuenta "Créditos en cuenta corriente"
"Créditos en cuentas a cobrar" corresponde a la cuenta "id:accounts.txt-1; content: cuenta "Cuentas por cobrar condicionadas"

"Cuentas por cobrar diversas" corresponde a la cuenta "Otras cuentas por cobrar"
"Cuentas por cobrar diversas a terceros" corresponde a la cuenta "Otras cuentas por cobrar"
"Deudores variosid:accounts.txt-1; content:" corresponde a la cuenta "Por riesgos operativos (Sociedad

In [58]:
response = client.vector_io.query(vector_db_id=VECTOR_DB_ID,
                       query=f'{ROOT_PROMPT}, prompt:{prompt}')

print(response)

{
  "chunks": [
    {
      "content": "\" corresponde a la cuenta \"Fideicomisos públicos de contratación\"\n\n\"Cuentas Varias\" corresponde a la cuenta \"Otros\"\n\"Cuentas Diversas\" corresponde a la cuenta \"Otros\"\n\"Otras Cuentas\" corresponde a la cuenta \"Otros",
      "metadata": {
        "token_count": 64.0,
        "document_id": "accounts.txt-1"
      }
    },
    {
      "content": "\" corresponde a la cuenta \"Por riesgos operativos (Sociedades de Información Crediticia)\"\n\"Cuenta de riesgos operativos en el balance de Sociedades de Información Crediticia\" corresponde a la cuenta \"Por riesgos operativos (Sociedades de Información Crediticia)\"\n\"Cuenta",
      "metadata": {
        "token_count": 64.0,
        "document_id": "accounts.txt-1"
      }
    },
    {
      "content": " en cuentas por pagar\" corresponde a la cuenta \"Créditos en cuenta corriente\"\n\"Créditos en cuentas corrientes\" corresponde a la cuenta \"Créditos en cuenta corriente\"\n\"Créditos e

### Definicion de Custom Tool y Prueba de ejecucion (fallido) ###

In [35]:
from typing import Dict, List, Union
from llama_stack_client.types.tool_def_param import Parameter
from llama_stack_client.types import ToolResponseMessage, UserMessage
from llama_stack_client import LlamaStackClient

from llama_stack_client.lib.agents.client_tool import ClientTool
import json

class QueryRAG(ClientTool):
    def __init__(self, client: LlamaStackClient, VECTOR_DB_ID: str):
        self.client = client
        self.VECTOR_DB_ID = VECTOR_DB_ID

    def get_name(self) -> str:
        return 'QueryRag'
    
    def get_description(self) -> str:
        return 'Obtiene la informacion mas relevante del RAG para enviar al contexto a partir de la query recibida.'
    
    def get_params_definition(self) -> Dict[str, Parameter]:
        return {
            "query": Parameter(name="query",
                               parameter_type="str",
                               description="Consulta a realizar en el RAG para traer contexto relevante",
                               required=True)
        }
    
    def run(self, query: str) -> str:
        """response = self.client.vector_io.query(vector_db_id=self.VECTOR_DB_ID,
                       query=query)"""
        
        print("Entre a run.")
        #response = {'query': f"Me insertaron: {query}"}

    
        #result = response.chunks[0].content
        return {"success": False, "error": "Cannot divide by zero."}

#@client_tool
'''
def query_rag(query: str) -> str:
    """
    Obtiene la informacion mas relevante del RAG para enviar al contexto a partir de la query recibida

    :param query: Consulta a realizar en el RAG. eg. 'Activo'
    :return: Contexto relevante a query.
    """

    response = client.vector_io.query(vector_db_id=VECTOR_DB_ID,
                       query=query)
    
    result = response.chunks[0].content

    return result'''

'\ndef query_rag(query: str) -> str:\n    """\n    Obtiene la informacion mas relevante del RAG para enviar al contexto a partir de la query recibida\n\n    :param query: Consulta a realizar en el RAG. eg. \'Activo\'\n    :return: Contexto relevante a query.\n    """\n\n    response = client.vector_io.query(vector_db_id=VECTOR_DB_ID,\n                       query=query)\n    \n    result = response.chunks[0].content\n\n    return result'

In [36]:

tools = [
        QueryRAG(client=client, VECTOR_DB_ID=VECTOR_DB_ID)
    ]

def inference_with_rag(prompt: str, root_prompt: str) -> Turn:
    available_shields = [shield.identifier for shield in client.shields.list()]

    root_prompt = "" if root_prompt == None else root_prompt
    agent_config = AgentConfig(
        model=MODEL,
        instructions=root_prompt,
        sampling_params={
            "strategy": {"type": "greedy"},
        },
        toolgroups=[
            """{
                "name": "builtin::rag",
                "args": {"vector_db_ids": [VECTOR_DB_ID]},
            }"""
        ],
        tool_choice="auto",
        tool_prompt_format="python_list",
        client_tools=[
            tool.get_tool_definition() for tool in tools
        ],
        input_shields=available_shields if available_shields else [],
        output_shields=available_shields if available_shields else [],
        enable_session_persistence=False,
    )

    agent = Agent(client, agent_config, tools)
    session_id = agent.create_session(SESSION_NAME)
    #print(f"Created session_id={session_id} for Agent({agent.agent_id})")

    response = agent.create_turn(
            messages=[
                UserMessage(role='user', content=prompt),
            ],
            session_id=session_id,
            stream=False
    )

    return response

In [37]:
from llama_stack_client.lib.agents.event_logger import EventLogger
response = inference_with_rag("Cuentas por cobrar de terceros", "")

KeyboardInterrupt: 

In [ ]:
response_copy = response
for log in EventLogger().log(response_copy):
    log.print()

AgentTurnResponseStreamChunk(event=TurnResponseEvent(payload=AgentTurnResponseTurnStartPayload(event_type='turn_start', turn_id='9afe0bf7-bc27-4b20-8a3f-f561e4b63c02')))
AgentTurnResponseStreamChunk(event=TurnResponseEvent(payload=AgentTurnResponseStepStartPayload(event_type='step_start', step_id='7113af50-b9a5-47d5-88d3-1ff7ca9e81be', step_type='inference', metadata={})))
inference> AgentTurnResponseStreamChunk(event=TurnResponseEvent(payload=AgentTurnResponseStepProgressPayload(delta=TextDelta(text='[', type='text'), event_type='step_progress', step_id='7113af50-b9a5-47d5-88d3-1ff7ca9e81be', step_type='inference')))
[AgentTurnResponseStreamChunk(event=TurnResponseEvent(payload=AgentTurnResponseStepProgressPayload(delta=TextDelta(text='Query', type='text'), event_type='step_progress', step_id='7113af50-b9a5-47d5-88d3-1ff7ca9e81be', step_type='inference')))
QueryAgentTurnResponseStreamChunk(event=TurnResponseEvent(payload=AgentTurnResponseStepProgressPayload(delta=TextDelta(text='R', t

TypeError: dump() missing 1 required positional argument: 'fp'

In [ ]:
client.tool_runtime.invoke_tool(
    tool_name="QueryRAG", kwargs={"query": "Cuentas por cobrar de terceros"}
)

BadRequestError: Error code: 400 - {'detail': 'Invalid value: Tool `QueryRAG` not served by any of the providers: brave-search, tavily-search, code-interpreter, rag-runtime, model-context-protocol. Make sure there is an Tools provider serving this tool.'}

### Canalizacion mediante informacion obtenida del RAG ###

In [71]:
client.vector_dbs.list()

[VectorDBListResponseItem(embedding_dimension=384, embedding_model='all-MiniLM-L6-v2', identifier='accounts', provider_id='faiss', provider_resource_id='accounts', type='vector_db'),
 VectorDBListResponseItem(embedding_dimension=384, embedding_model='all-MiniLM-L6-v2', identifier='cyber', provider_id='faiss', provider_resource_id='cyber', type='vector_db'),
 VectorDBListResponseItem(embedding_dimension=384, embedding_model='all-MiniLM-L6-v2', identifier='demo-app-gradio', provider_id='faiss', provider_resource_id='demo-app-gradio', type='vector_db'),
 VectorDBListResponseItem(embedding_dimension=384, embedding_model='all-MiniLM-L6-v2', identifier='interio_bank', provider_id='faiss', provider_resource_id='interio_bank', type='vector_db'),
 VectorDBListResponseItem(embedding_dimension=384, embedding_model='all-MiniLM-L6-v2', identifier='memory-bank', provider_id='faiss', provider_resource_id='memory-bank', type='vector_db'),
 VectorDBListResponseItem(embedding_dimension=384, embedding_mo

In [97]:
account = '"Conversión a elección de la empresa emisora"'                       # No la trae el contexto - existe en el RAG.
account = '"Obligaciones financieras"'                                          # Aparece multiples veces en el RAG
account = '"Deudas con terceros y otras obligaciones"'                          # No la trae el contexto - existe en el RAG.
account = '"Inmuebles en posesión del banco"'                                   # No la trae el contexto - existe en el RAG.
account = '"Agotamientos"'                                                      # No la trae el contexto - existe en el RAG.
account = '"Activos financieros de la compañía"'                                # No la trae el contexto - existe en el RAG.
account = '"Obligaciones a largo plazo"'                                        # Aparece multiples veces en el RAG
account = '"Ganancias de acciones que cumplen con los requisitos de capital"'   # No la trae el contexto - existe en el RAG.
account = '"Transacciones relacionadas con créditos"'                           # No la trae el contexto - existe en el RAG
account = '"Instituciones Crediticias"'                                         # No la trae el contexto - existe en el RAG
account = '"Instrumentos financieros de capital"'                               # Aparece multiples veces en el RAG
account = '"Obligaciones fiscales por utilidades" corresponde a la cuenta "'                              # No la trae el contexto - existe en el RAG

response = response = client.vector_io.query(vector_db_id=VECTOR_DB_ID,
                       query=account)
pprint(response.to_dict())
print(response.chunks[0].content)

{'chunks': [{'content': 'idad"\n'
                        '"Pasivo fiscal por utilidades" corresponde a la '
                        'cuenta "Pasivo por impuestos a la utilidad"\n'
                        '"Responsabilidades tributarias por ganancias" '
                        'corresponde a la cuenta "Pasivo por impuestos a la '
                        'utilidad"\n'
                        '\n'
                        '"Impuestos por pagar" corresponde a la cuenta',
             'metadata': {'document_id': 'accounts.txt-1',
                          'token_count': 64.0}},
            {'content': ' obligaciones fiscales" corresponde a la cuenta '
                        '"Créditos fiscales"\n'
                        '\n'
                        '"Provisión por impuestos diferidos no recuperables" '
                        'corresponde a la cuenta "Estimación por impuestos a '
                        'la utilidad diferidos no recuperables"\n'
                        '"Reserva para impu

In [13]:
account = 'Bienes'
prompt = prompt_generator(account)
response = inference_with_rag(prompt)
pprint(response.to_dict())

{'completed_at': datetime.datetime(2025, 2, 27, 17, 15, 26, 347165),
 'input_messages': [{'content': 'Dispones del siguiente contexto ``` de bienes '
                                'corresponde a la cuenta Resultado por '
                                'adjudicación de bienes.\n'
                                'Ganancias por venta de bienes corresponde a '
                                'la cuenta Resultado por adjudicación de '
                                'bienes.\n'
                                'Utilidades por adjudicación de activos '
                                'corresponde a la cuenta Resultado por '
                                'adjudicación de bienes```\n'
                                '    \n'
                                '    Tu tarea es responder a que cuenta '
                                'corresponde la siguiente: ```Bienes```\n'
                                '    ',
                     'context': [{'text': 'Here are the retrieved documents '
  

In [119]:
account = 'Reservas patrimoniales'

response = response = client.vector_io.query(vector_db_id=VECTOR_DB_ID,
                       query=account)
pprint(response.to_dict())

{'chunks': [{'content': ' con reserva de dominio corresponde a la cuenta '
                        'Cobros anticipados de bienes prometidos en venta o '
                        'con reserva de dominio.\n'
                        'Ingresos anticipados por bienes pendientes de entrega '
                        'corresponde a la cuenta Cobros anticipados de bienes '
                        'prometidos en venta o con reserva de dominio',
             'metadata': {'document_id': 'accounts.txt-1',
                          'token_count': 64.0}},
            {'content': ' reserva de dominio.\n'
                        'Cobros anticipados por mercancías en reserva '
                        'corresponde a la cuenta Cobros anticipados de bienes '
                        'prometidos en venta o con reserva de dominio.\n'
                        'Adelantos por bienes sujetos a dominio reservado '
                        'corresponde a la cuenta Cobros anticipados de',
             'metadata': {'doc